# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faja27/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Turns the Week 4-6 work (baseline → model → validated, grouped-split model) into something a content team could actually act on next Monday.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Archetype → action mapping.** Rather than one flat score, the validated model's probability (Week 5-6, grouped-split random forest) sorts every candidate into one of three archetypes:

| archetype | model probability | action | reason code |
|---|---|---|---|
| `quick_win_high_confidence` | ≥ 0.70 | prioritize for CTR/content fix this cycle | `high_confidence_opportunity` |
| `quick_win_borderline` | 0.40 - 0.70 | queue for a human look, not auto-actioned | `borderline_needs_review` |
| `already_efficient` | < 0.40 | no action, monitor only | `already_efficient` |
| *(outside the lane)* | n/a | no action, out of scope | `not_in_lane` |

Within each archetype, pages are ranked by `action_priority_score = model_probability × impressions_90d` combining *how likely* the page is a real opportunity with *how many people would benefit* from fixing it. Pure volume (Week 4) or pure probability alone would each miss half the picture; this multiplies them.

**Decay/refresh insight:** within the candidate pool, the underperformance rate climbs with content age - 41.8% (0-90 days) → 66.9% (91-180d) → 68.7% (181-365d) → **78.3% for 365+ day content**. Older content in this lane is measurably more likely to need a fix, not just more likely to exist - a concrete, data-backed argument for a refresh cadence, not just a one-time push.

**Cost/value framing:** `quick_win_high_confidence` is where review effort earns the most - roughly 84% of that bucket is a real opportunity, so a reviewer's time there converts to fixes at a high hit rate. The `borderline` bucket sits close to a coin flip (see section 3) - reviewing it is still worthwhile because of its size, but each review costs the same regardless of the bucket, so it's the lower-value use of a reviewer's hour.
`already_efficient` is intentionally small and mostly used to confirm the model isn't flagging good pages.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faja27/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

SEED = 42
pd.set_option("display.width", 140)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --- reproduce the lane, label, and grouped-split model exactly as validated in weeks 4-6 ---
in_range = df["position_tier"].isin(["page_1", "striking"])
has_demand = df["impression_tier"].isin(["moderate", "good", "excellent"])
is_candidate = in_range & has_demand
df["quick_win_score"] = np.where(is_candidate, df["impressions_90d"], 0)
cand = df[is_candidate].copy().reset_index(drop=True)

bench_pool = df[df["position_tier"].isin(["page_1", "striking"])]
benchmark_ctr = 100 * bench_pool["clicks_90d"].sum() / bench_pool["impressions_90d"].sum()
cand["label"] = (cand["ctr"] < benchmark_ctr).astype(int)
cand["has_keyword_data"] = cand["search_volume"].notna().astype(int)

num_feats = ["search_volume", "competition", "cpc", "word_count", "char_count",
             "content_age_days", "days_since_last_update", "impressions_90d",
             "engagement_rate", "scroll_rate", "ai_traffic_pct", "avg_position", "has_keyword_data"]
cat_feats = ["competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
             "position_tier", "impression_tier"]

X = cand[num_feats + cat_feats].copy()
for c in cat_feats:
    X[c] = X[c].fillna("unknown")
y = cand["label"].values
groups = cand["client_id"].values

pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_feats),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats),
])
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, _ = next(gss.split(X, y, groups))  # train on the same grouped train split validated in week 6

rf = Pipeline([("pre", pre), ("clf", RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=SEED, n_jobs=-1))])
rf.fit(X.iloc[train_idx], y[train_idx])
cand["model_proba"] = rf.predict_proba(X)[:, 1]

# --- decay/refresh insight ---
cand["age_bucket"] = pd.cut(cand["content_age_days"], bins=[0, 90, 180, 365, 100000],
                             labels=["0-90d", "91-180d", "181-365d", "365d+"])
decay_table = cand.groupby("age_bucket", observed=True).agg(n=("content_id", "size"), label_rate=("label", "mean"))
print("decay/refresh insight (underperformance rate by content age):")
print(decay_table)

# --- archetype -> action mapping ---
def archetype(p):
    if p >= 0.70:
        return "quick_win_high_confidence", "prioritize_for_ctr_fix", "high_confidence_opportunity"
    elif p >= 0.40:
        return "quick_win_borderline", "queue_for_review", "borderline_needs_review"
    else:
        return "already_efficient", "no_action_monitor", "already_efficient"

cand[["archetype", "action", "reason_code"]] = cand["model_proba"].apply(lambda p: pd.Series(archetype(p)))
cand["action_priority_score"] = cand["model_proba"] * cand["impressions_90d"]

print("\narchetype sizes and calibration (actual label rate should roughly match the probability band):")
print(cand.groupby("archetype").agg(n=("content_id", "size"), avg_model_proba=("model_proba", "mean"),
                                     actual_label_rate=("label", "mean")))

ranked_queue = (cand.sort_values("action_priority_score", ascending=False)
                [["content_id", "client_id", "archetype", "action", "reason_code",
                  "model_proba", "impressions_90d", "action_priority_score",
                  "position_tier", "avg_position", "ctr", "content_age_days"]]
                .reset_index(drop=True))
ranked_queue.insert(0, "rank", ranked_queue.index + 1)
print(f"\ntop of the queue:")
print(ranked_queue.head(5).to_string(index=False))


decay/refresh insight (underperformance rate by content age):
               n  label_rate
age_bucket                  
0-90d        196    0.418367
91-180d     5373    0.669458
181-365d    4158    0.687350
365d+       2974    0.782784

archetype sizes and calibration (actual label rate should roughly match the probability band):
                              n  avg_model_proba  actual_label_rate
archetype                                                          
already_efficient           284         0.362285           0.091549
quick_win_borderline       4896         0.585540           0.516544
quick_win_high_confidence  7521         0.801605           0.838984

top of the queue:
 rank           content_id         client_id                 archetype                 action                 reason_code  model_proba  impressions_90d  action_priority_score position_tier  avg_position  ctr  content_age_days
    1 content_aaef01a50def client_19581e27de quick_win_high_confidence prioritize_f

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** a content/SEO lead's weekly triage list — which already-close-to-page-1 pages to look at first for a CTR or content fix, ranked by expected payoff. It's a **prioritization aid**, not a publish/no-publish decision-maker.

**Limits, stated plainly:**
- **Narrow lane, by design.** This only covers pages already in `page_1`/`striking` position with `moderate`+
  demand (see the coverage number below) — it says nothing about deep pages, zero-demand pages, or brand-new
  content.
- **Trained on a handful of clients.** The validated split (Week 6) covered 23 training clients / 6 held-out —
  a brand-new client outside that mix is untested, not just "less certain."
- **Cross-sectional, not causal.** The model ranks who looks like an opportunity *in this snapshot*; it does
  not claim that fixing a page *will* produce a given lift (see Week-6's claim rewrite for why).
- **A composite validation metric, not a guarantee.** Precision@K (Week 5-6) was measured on a held-out slice
  of this same dataset - real-world hit rate on a live queue should be tracked, not assumed identical.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

coverage_pct = len(cand) / len(df)
print(f"lane coverage: {len(cand):,} of {len(df):,} total content pieces ({coverage_pct:.1%})")
print(f"training clients in the validated split: {cand.iloc[train_idx]['client_id'].nunique()} "
      f"of {cand['client_id'].nunique()} total clients in this lane")
print("\nplain statement for the playbook's front page:")
print(f"  this playbook ranks {coverage_pct:.0%} of the portfolio (the page_1/striking, moderate+ demand slice).")
print(f"  the other {1 - coverage_pct:.0%} needs a different lane's analysis, not this queue.")

lane coverage: 12,701 of 30,000 total content pieces (42.3%)
training clients in the validated split: 23 of 29 total clients in this lane

plain statement for the playbook's front page:
  this playbook ranks 42% of the portfolio (the page_1/striking, moderate+ demand slice).
  the other 58% needs a different lane's analysis, not this queue.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any row, a human checks:**
- Query/intent match does the page actually target a query worth winning, or is this a stray impression?
- Recent manual activity was this page just promoted, A/B tested, or seasonally spiked? The model can't see
  that context.
- Brand/pillar fit does a CTR/content fix here match the client's current content strategy?

**No-go list — never automate these:**
- **Never auto-publish or auto-edit content** from this queue. It's a review list, not a CMS action.
- **Never treat `already_efficient` as "suppress" or "deprioritize."** Low probability of being an opportunity
  is not evidence a page is *overperforming* or should be touched at all - it's evidence to leave it alone.
- **Never act on `quick_win_borderline` without a human look** - the code cell below shows why: its actual
  label rate sits close to a coin flip, so treating it like `high_confidence` would roughly halve the hit rate.
- **Never present this queue's ranking as a client-facing guarantee** ("fixing this WILL grow traffic by X%")
  - see the intended-use limits above.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

review_stats = cand.groupby("archetype")["label"].agg(n="size", hit_rate="mean")
print(review_stats)

coin_flip_gap = abs(review_stats.loc["quick_win_borderline", "hit_rate"] - 0.5)
print(f"\n'quick_win_borderline' hit rate is {review_stats.loc['quick_win_borderline','hit_rate']:.1%} - "
      f"{coin_flip_gap:.1%} away from a 50/50 coin flip. That's the concrete reason it's a 'queue for review'")
print("bucket, not an auto-action bucket like 'high_confidence'.")

print(f"\n'already_efficient' bucket is small (n={review_stats.loc['already_efficient','n']}) with a low "
      f"{review_stats.loc['already_efficient','hit_rate']:.1%} hit rate - thin evidence either way, which is")
print("exactly why it's a monitor-only bucket and not a candidate for any kind of automated deprioritization.")


                              n  hit_rate
archetype                                
already_efficient           284  0.091549
quick_win_borderline       4896  0.516544
quick_win_high_confidence  7521  0.838984

'quick_win_borderline' hit rate is 51.7% - 1.7% away from a 50/50 coin flip. That's the concrete reason it's a 'queue for review'
bucket, not an auto-action bucket like 'high_confidence'.

'already_efficient' bucket is small (n=284) with a low 9.2% hit rate - thin evidence either way, which is
exactly why it's a monitor-only bucket and not a candidate for any kind of automated deprioritization.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Light, checkable signals - not a full MLOps pipeline, just what a human re-running this quarterly should
watch for:

- **Base rate drift.** Current test-set base rate is ~0.68-0.70 (Week 5-6). If a future quarter's candidate
  pool shows a base rate far outside that range, the benchmark CTR itself has probably shifted - re-derive it,
  don't reuse the old cutoff blindly.
- **Calibration drift.** The archetype hit rates below (84% / 52% / 9%) are the reference point. If a new
  quarter's `quick_win_high_confidence` bucket hit rate drops materially (e.g. below ~70%), the model's
  probability estimates are no longer trustworthy at face value - retrain before shipping the queue.
- **Coverage drift.** If the lane's coverage (currently ~42% of the portfolio) swings sharply, the underlying
  position/impression distribution has shifted enough that the whole lane definition should be revisited, not
  just the model inside it.
- **New-client blind spot.** Per Week 6's grouped-split finding, this model is validated on a subset of
  clients. Any brand-new client's queue rows should be flagged as "unvalidated for this client" until enough
  of their data has gone through at least one full review cycle.
- **Retrain cadence:** quarterly by default, or immediately if any trigger above fires early.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

monitoring_snapshot = {
    "generated_from": "w07_action_playbook.ipynb",
    "benchmark_ctr_pct": round(benchmark_ctr, 3),
    "test_base_rate_range": [0.68, 0.70],
    "lane_coverage_pct": round(coverage_pct, 3),
    "archetype_hit_rates": review_stats["hit_rate"].round(3).to_dict(),
    "archetype_sizes": review_stats["n"].to_dict(),
    "training_clients": int(cand.iloc[train_idx]["client_id"].nunique()),
    "total_lane_clients": int(cand["client_id"].nunique()),
}
print("monitoring snapshot (this is the reference point future runs get diffed against):")
for k, v in monitoring_snapshot.items():
    print(f"  {k}: {v}")


monitoring snapshot (this is the reference point future runs get diffed against):
  generated_from: w07_action_playbook.ipynb
  benchmark_ctr_pct: 0.35
  test_base_rate_range: [0.68, 0.7]
  lane_coverage_pct: 0.423
  archetype_hit_rates: {'already_efficient': 0.092, 'quick_win_borderline': 0.517, 'quick_win_high_confidence': 0.839}
  archetype_sizes: {'already_efficient': 284, 'quick_win_borderline': 4896, 'quick_win_high_confidence': 7521}
  training_clients: 23
  total_lane_clients: 29


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Queue CSV goes to `work/outputs/` (regenerated each run, not committed - same leak-guard convention as Week 4). The monitoring snapshot goes to `work/outputs/*.json` (committed - it's the receipt the paper's numbers trace back to). One figure - the decay/refresh chart - goes to `work/figures/` (committed, reused
in the paper).

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 1. the ranked queue CSV (not committed - regenerated every run)
queue_path = "work/outputs/action_playbook_queue.csv"
ranked_queue.to_csv(queue_path, index=False)
print(f"wrote {len(ranked_queue):,} rows to {queue_path}")

# 2. the monitoring snapshot JSON (committed - the paper's numbers trace back to this)
metrics_path = "work/outputs/w07_monitoring_snapshot.json"
with open(metrics_path, "w") as f:
    json.dump(monitoring_snapshot, f, indent=2)
print(f"wrote monitoring snapshot to {metrics_path}")

# 3. the decay/refresh figure (committed - reused in the paper)
fig, ax = plt.subplots(figsize=(6, 4))
decay_table["label_rate"].plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_ylabel("share underperforming benchmark CTR")
ax.set_xlabel("content age")
ax.set_title("Content age vs. underperformance rate\n(decay/refresh insight)")
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
fig_path = "work/figures/w07_decay_refresh_insight.png"
plt.savefig(fig_path, dpi=150)
plt.close(fig)
print(f"wrote figure to {fig_path}")

print("\nreminder: the CSV in work/outputs/ stays out of git by design (CI leak-guard blocks data files).")
print("the JSON and the figure ARE meant to be committed - they're the receipts the paper builds on.")

wrote 12,701 rows to work/outputs/action_playbook_queue.csv
wrote monitoring snapshot to work/outputs/w07_monitoring_snapshot.json
wrote figure to work/figures/w07_decay_refresh_insight.png

reminder: the CSV in work/outputs/ stays out of git by design (CI leak-guard blocks data files).
the JSON and the figure ARE meant to be committed - they're the receipts the paper builds on.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.